# Step 1 check 3 - Structure below the GCM cell: maps, domain and PRUDENCE regions

**Ad hoc test, outside the chain**: it reads the step-1 product of the supervisor's pipeline (EC-Earth3 climate
imprint + QDM at 0.25 deg, `qdm/merged`) and ERA5 at 0.25 deg (`guido_inputs`), and writes only a cache on EOS
(`outputs/step1_check/cache/`) and figures / tables in `outputs/` next to this notebook. It complements notebook 1,
whose spatial statistics (correlation of the anomalies against distance up to 1500 km) are dominated by the
large-scale weather that the product inherits from EC-Earth3, and which uses 3000 random pixels only.

**Question.** What does the product miss below the GCM cell (about 0.7 deg, 3 x 3 pixels at 0.25 deg), where, and is
it more than the sampling noise? This is what the analogue block (block 3 of the pipeline) could correct. Below the
GCM cell the product has the GCM anomaly interpolated bilinearly (tasmax, dtr) or the pattern of a reanalysis day (pr),
plus the fine-scale climatology of the imprint.

**Method.** Calibration years `YEARS` (1981-2014), all pixels of the LISFLOOD domain (`ba_utils.load_domain_mask`),
four seasons. Variables: tasmax, dtr, pr (when its product exists) and tasmin = tasmax - dtr (what LISFLOOD receives).
For each pixel and day:
- **anomaly** a = value - mean of the same calendar month of the same year (day-to-day variability: the seasonal cycle,
  the interannual variability and the trend are left out);
- **mean of the GCM cell** c = mean of the domain pixels whose centre falls in the EC-Earth3 cell of the pixel;
- **departure** d = a - (anomaly of c): the part of the daily anomaly below the GCM cell.

| statistic | what it measures | expected in the product |
|---|---|---|
| `sd_dep` SD of the departure (relative difference) | daily variability below the GCM cell | lower (smooth interpolation) |
| `corr_cell` correlation of the pixel with its GCM-cell mean | how much the pixel follows the cell | higher (close to 1) |
| `corr_dep_k` correlation of the departure with the pixel 28, 56, 83, 139 km north | spatial scale of the structure below the cell | different: the departures of a bilinear interpolation are ramps inside each cell that change sign at its edges (often lower at 28 km) |
| `dep_mean` mean departure from the GCM-cell mean (raw values) | fine-scale climatological pattern | close to ERA5 (the imprint is built from it) |
| `lag1` lag-1 autocorrelation of the anomaly (days of the same month) | persistence | from EC-Earth3 |
| `corr_pair` correlation of the anomalies of two variables | dependence between the variables | from EC-Earth3 |

For `lag1` and `corr_pair` the same statistic of the GCM-cell mean series gives the error at the GCM scale, and the
**gap** = difference at 0.25 deg - difference at the GCM cell: the part created below the cell.

**Noise.** ERA5 is split three times into two halves of 17 years (1981-1997 / 1998-2014, even / odd years, a mixed
split); the difference between the halves, scaled to the full period, gives the size of a difference between two
independent 34-year samples. t = (product - ERA5) / noise; |t| > `T_CRIT` (Student, 3 degrees of freedom, 5 %) is
**significant**, at each pixel (maps, share of the pixels) and for the regional means. For `dep_mean` the regional
value is the RMS of the pattern and the difference is the RMS of (product - ERA5).

Outputs: table `subgrid_regions_<TAG>.csv` (domain and PRUDENCE regions), maps per statistic and variable
(ERA5 | product | difference | t | gap), correlation of the departures against distance per region, and one summary
heatmap per variable (colour = t, text = difference).

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, glob, time, hashlib, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import sparse, stats
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None
cfg = bu.load_config(CONFIG_PATH)
RUN = os.environ.get("STEP1_RUN", "stab2050nohos")          # run folder of the pipeline (its 1981-2014 part is checked)
BASE = Path(cfg["data_root"]) / "elisa" / "GCM_downscaling"
RUN_DIRS = BASE / "downscaling"                               # run folders created by guido_pipeline/setup_run.sh
INP = BASE / "guido_inputs"                                   # inputs written by guido_pipeline/prepare_inputs.sh
ALL_VARS = ["tasmax", "dtr", "pr"]                            # tasmin = tasmax - dtr is added when both exist
YEARS = (1981, 2014)
SPLIT_YEAR = 1997                                             # halves 1981-1997 / 1998-2014 (noise of ERA5)
SEASONS = {"DJF": (12, 1, 2), "MAM": (3, 4, 5), "JJA": (6, 7, 8), "SON": (9, 10, 11)}
SHIFTS = (1, 2, 3, 5)                                         # neighbour 1, 2, 3, 5 pixels north: 28, 56, 83, 139 km
MIN_PIX_CELL = 4                                              # GCM cells with fewer domain pixels: no departure statistic
PRUDENCE = {"BI": ("British Isles", (50, 59), (-10, 2)), "IP": ("Iberian Peninsula", (36, 44), (-10, 3)),
            "FR": ("France", (44, 50), (-5, 5)), "ME": ("Mid-Europe", (48, 55), (2, 16)),
            "SC": ("Scandinavia", (55, 70), (5, 30)), "AL": ("Alps", (44, 48), (5, 15)),
            "MD": ("Mediterranean", (36, 44), (3, 25)), "EA": ("Eastern Europe", (44, 55), (16, 30))}   # lat, lon boxes
OVERWRITE = False

CACHE = bu.out_dir(cfg, "step1_check", "cache")
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
TAG = f"{RUN}_{YEARS[0]}-{YEARS[1]}"
print(f"run {RUN} | years {YEARS[0]}-{YEARS[1]} | cache {CACHE}")

## 1. Files, grid, domain, GCM cells, neighbours, regions

In [ ]:
def product_file(var):
    run_dir = RUN_DIRS / f"climdown_cdo_{var}_europe_EC-Earth3-{RUN}_stab"
    fs = sorted(glob.glob(str(run_dir / "data" / "*" / "out" / "qdm" / "merged" / f"*01_12*{var}*.nc")))
    fs = [f for f in fs if "unique_qdm_" not in Path(f).name]      # monthly merges of postprocess_qdm, not the product
    if len(fs) != 1:
        print(f"  {var}: {len(fs)} product files in {run_dir}/data/*/out/qdm/merged: {[Path(f).name for f in fs]}")
        return None
    return Path(fs[0])


FILES = {v: {"ERA5": INP / "era5" / f"{v}_era5_1981_2014_europe.nc",
             "GCM": INP / RUN / f"{v}_EC-Earth3-{RUN}_HISTORICAL_europe.nc",
             "PROD": product_file(v)} for v in ALL_VARS}
BASE_VARS = [v for v in ALL_VARS if FILES[v]["PROD"] is not None and FILES[v]["ERA5"].exists() and FILES[v]["GCM"].exists()]
for v in ALL_VARS:
    print(f"{v:6s} " + " | ".join(f"{k}: {p.name if p is not None and p.exists() else 'MISSING'}" for k, p in FILES[v].items()))
assert BASE_VARS, "no step-1 product yet"
VARS = BASE_VARS + (["tasmin"] if {"tasmax", "dtr"} <= set(BASE_VARS) else [])
PAIRS = [(a, b) for i, a in enumerate(VARS) for b in VARS[i + 1:]]
print("variables:", VARS, "| pairs:", [f"{a}-{b}" for a, b in PAIRS])


def open_var(path, var):
    # daily field of YEARS, standard coordinates (ascending lat, lon -180..180), singleton dimensions and 29 February dropped
    ds = xr.open_dataset(path, chunks={"time": 365}, **bu._TIME_KW)
    da = ds[var] if var in ds.data_vars else ds[list(ds.data_vars)[0]]
    for d in list(da.dims):
        if d not in ("time", "lat", "lon", "latitude", "longitude", "y", "x") and da.sizes[d] == 1:
            da = da.isel({d: 0}, drop=True)
    da = bu.norm_coords(da).transpose("time", "lat", "lon")
    t = da.indexes["time"]
    keep = (t.year >= YEARS[0]) & (t.year <= YEARS[1]) & ~((t.month == 2) & (t.day == 29))
    return da.isel(time=np.flatnonzero(keep))


F0 = open_var(FILES[BASE_VARS[0]]["ERA5"], BASE_VARS[0])
G0 = open_var(FILES[BASE_VARS[0]]["GCM"], BASE_VARS[0])
T_YEAR = np.asarray(F0.indexes["time"].year); T_MONTH = np.asarray(F0.indexes["time"].month)
NT = T_YEAR.size
assert NT == 365 * (YEARS[1] - YEARS[0] + 1), f"{NT} days in ERA5"
FLAT, FLON, GLAT, GLON = F0.lat.values, F0.lon.values, G0.lat.values, G0.lon.values

# 0.25 deg pixel -> EC-Earth3 cell containing its centre; domain pixels P; mean of the domain pixels of each cell
ye, xe = bu._edges(GLAT), bu._edges(GLON)
iy = np.searchsorted(ye, FLAT) - 1; ix = np.searchsorted(xe, FLON) - 1
CELL = np.where(((iy >= 0) & (iy < GLAT.size))[:, None] & ((ix >= 0) & (ix < GLON.size))[None, :],
                iy[:, None] * GLON.size + ix[None, :], -1).ravel()
DOM = bu.load_domain_mask(cfg, "era5", like=F0.isel(time=0)).values.ravel() & (CELL >= 0)
P = np.flatnonzero(DOM); NP = P.size
_, CPOS, CNT = np.unique(CELL[P], return_inverse=True, return_counts=True)
AGG = sparse.csr_matrix((1.0 / CNT[CPOS], (CPOS, np.arange(NP))), shape=(CNT.size, NP))
OK_CELL = CNT[CPOS] >= MIN_PIX_CELL

# neighbour k pixels north (same column), inside the domain
POS = np.full(FLAT.size * FLON.size, -1); POS[P] = np.arange(NP)
ROW, COL = P // FLON.size, P % FLON.size
NB = {k: np.where(ROW + k < FLAT.size, POS[np.minimum(ROW + k, FLAT.size - 1) * FLON.size + COL], -1) for k in SHIFTS}
VALID_NB = {k: (NB[k] >= 0) & OK_CELL & OK_CELL[np.maximum(NB[k], 0)] for k in SHIFTS}
DIST = {k: k * float(np.diff(FLAT).mean()) * 111.2 for k in SHIFTS}

PLAT, PLON = FLAT[ROW], FLON[COL]
WGT = np.cos(np.deg2rad(PLAT))
REGIONS = {"DOMAIN": np.ones(NP, bool),
           **{r: (PLAT >= la[0]) & (PLAT <= la[1]) & (PLON >= lo[0]) & (PLON <= lo[1]) for r, (_, la, lo) in PRUDENCE.items()}}
print(f"{NP} domain pixels in {CNT.size} GCM cells ({OK_CELL.mean():.0%} of the pixels in cells with >= {MIN_PIX_CELL}); "
      + ", ".join(f"{r} {m.sum()}" for r, m in REGIONS.items()))

## 2. Accumulation (one year at a time, cached on EOS)

For each dataset, group of years (first / second half x even / odd year), season and pixel: the sums of the products
needed by the statistics (anomalies of the pixel and of its GCM-cell mean, departures of neighbours, consecutive days,
pairs of variables). The cache name contains the modification times of the files read: a new product file is read
again automatically.

In [ ]:
SEAS_OF_MONTH = {m: i for i, ms in enumerate(SEASONS.values()) for m in ms}
NS = len(SEASONS)
KEYS = ["Saa", "Scc", "Sac", "Spa", "Spc", "Sdep"] + [f"Sd{k}" for k in SHIFTS]


def group(y):
    return 2 * int(y > SPLIT_YEAR) + y % 2                    # 0..3: first / second half x even / odd year


def stamp(name):
    s = ";".join(f"{FILES[v][name]}:{int(os.path.getmtime(FILES[v][name]))}" for v in BASE_VARS)
    return hashlib.md5(s.encode()).hexdigest()[:8]


def accumulate(name):
    path = CACHE / f"subgrid_{name}_{'-'.join(BASE_VARS)}_{TAG}_{stamp(name)}.npz"
    if path.exists() and not OVERWRITE:
        return dict(np.load(path))
    t0 = time.time()
    das = {v: open_var(FILES[v][name], v) for v in BASE_VARS}
    for v, da in das.items():
        assert da.sizes["time"] == NT, f"{name} {v}: {da.sizes['time']} days"
        assert bu.same_grid(da, F0), f"{name} {v}: not on the ERA5 0.25 deg grid"
    acc = {f"{k}_{v}": np.zeros((4, NS, NP)) for v in VARS for k in KEYS}
    acc.update({f"{k}_{a}_{b}": np.zeros((4, NS, NP)) for a, b in PAIRS for k in ("Sxy", "Sxyc")})
    acc["n"] = np.zeros((4, NS)); acc["npair"] = np.zeros((4, NS))
    for y in range(YEARS[0], YEARS[1] + 1):
        ti = np.flatnonzero(T_YEAR == y); q = group(y); mon = T_MONTH[ti]
        X = {v: das[v].isel(time=ti).values.reshape(ti.size, -1)[:, P].astype("float64") for v in BASE_VARS}
        if "tasmin" in VARS:
            X["tasmin"] = X["tasmax"] - X["dtr"]
        for m in range(1, 13):
            i = np.flatnonzero(mon == m); s = SEAS_OF_MONTH[m]
            acc["n"][q, s] += i.size; acc["npair"][q, s] += i.size - 1
            A, AC = {}, {}
            for v in VARS:
                x = X[v][i]
                c = np.asarray(AGG @ x.T).T[:, CPOS]                 # mean of the domain pixels of the GCM cell
                acc[f"Sdep_{v}"][q, s] += (x - c).sum(0)
                a = x - x.mean(0); ac = c - c.mean(0); d = a - ac     # anomalies of this month of this year, departure
                acc[f"Saa_{v}"][q, s] += (a * a).sum(0)
                acc[f"Scc_{v}"][q, s] += (ac * ac).sum(0)
                acc[f"Sac_{v}"][q, s] += (a * ac).sum(0)
                acc[f"Spa_{v}"][q, s] += (a[:-1] * a[1:]).sum(0)
                acc[f"Spc_{v}"][q, s] += (ac[:-1] * ac[1:]).sum(0)
                for k in SHIFTS:
                    ok = NB[k] >= 0
                    acc[f"Sd{k}_{v}"][q, s, ok] += (d[:, ok] * d[:, NB[k][ok]]).sum(0)
                A[v], AC[v] = a, ac
            for a_, b_ in PAIRS:
                acc[f"Sxy_{a_}_{b_}"][q, s] += (A[a_] * A[b_]).sum(0)
                acc[f"Sxyc_{a_}_{b_}"][q, s] += (AC[a_] * AC[b_]).sum(0)
        if (y - YEARS[0]) % 5 == 4:
            print(f"  {name}: {y} ({time.time() - t0:.0f} s)")
    np.savez(path, **acc)
    print(f"  {name}: done ({time.time() - t0:.0f} s) -> {path.name}")
    return acc


ACC = {name: accumulate(name) for name in ("ERA5", "PROD")}

## 3. Statistics per pixel, noise, tables for the domain and the PRUDENCE regions

In [ ]:
SPLITS = {"halves": ((0, 1), (2, 3)), "even/odd": ((0, 2), (1, 3)), "mixed": ((0, 3), (1, 2))}
T_CRIT = float(stats.t.ppf(0.975, len(SPLITS)))
STATS = {"sd_dep": ("SD of the daily departure from the GCM-cell mean", "rel", None),
         "corr_cell": ("correlation of the pixel with the mean of its GCM cell", "abs", None),
         **{f"corr_dep_{k}": (f"correlation of the departures with the pixel {DIST[k]:.0f} km north", "abs", None)
            for k in SHIFTS},
         "dep_mean": ("mean departure from the GCM-cell mean (fine-scale climatology)", "pattern", None),
         "lag1": ("lag-1 autocorrelation of the daily anomaly", "abs", "lag1_cell"),
         "corr_pair": ("correlation of the daily anomalies of the two variables", "abs", "corr_pair_cell")}
TARGETS = [(st, v) for st in STATS if st != "corr_pair" for v in VARS] + [("corr_pair", f"{a}-{b}") for a, b in PAIRS]


def pixel_stats(acc, groups):
    g = list(groups)
    tot = lambda key: acc[key][g].sum(0)
    n = tot("n")[:, None]; npair = tot("npair")[:, None]
    S = {}
    with np.errstate(invalid="ignore", divide="ignore"):
        for v in VARS:
            saa, scc, sac = tot(f"Saa_{v}"), tot(f"Scc_{v}"), tot(f"Sac_{v}")
            sdd = np.maximum(saa - 2 * sac + scc, 0)
            S["sd_dep", v] = np.sqrt(sdd / n)
            S["corr_cell", v] = sac / np.sqrt(saa * scc)
            for k in SHIFTS:
                r = np.full_like(sdd, np.nan); ok = VALID_NB[k]
                r[:, ok] = tot(f"Sd{k}_{v}")[:, ok] / np.sqrt(sdd[:, ok] * sdd[:, NB[k][ok]])
                S[f"corr_dep_{k}", v] = r
            S["dep_mean", v] = tot(f"Sdep_{v}") / n
            for key in ("sd_dep", "corr_cell", "dep_mean"):
                S[key, v][:, ~OK_CELL] = np.nan
            S["lag1", v] = (tot(f"Spa_{v}") / npair) / (saa / n)
            S["lag1_cell", v] = (tot(f"Spc_{v}") / npair) / (scc / n)
        for a_, b_ in PAIRS:
            S["corr_pair", f"{a_}-{b_}"] = tot(f"Sxy_{a_}_{b_}") / np.sqrt(tot(f"Saa_{a_}") * tot(f"Saa_{b_}"))
            S["corr_pair_cell", f"{a_}-{b_}"] = tot(f"Sxyc_{a_}_{b_}") / np.sqrt(tot(f"Scc_{a_}") * tot(f"Scc_{b_}"))
    return S


PS = {name: pixel_stats(ACC[name], (0, 1, 2, 3)) for name in ("ERA5", "PROD")}
HALF = {(sp, j): pixel_stats(ACC["ERA5"], gg[j]) for sp, gg in SPLITS.items() for j in (0, 1)}


def contrast(a, b, kind):
    with np.errstate(invalid="ignore", divide="ignore"):
        return a / b - 1.0 if kind == "rel" else a - b


def noise(D):
    # D: contrasts between two halves of 17 years -> SD of a difference between two independent 34-year samples
    return np.sqrt(sum(d ** 2 for d in D) / len(D) / 2)


# per pixel: difference, noise, t (and the difference of the GCM-cell series)
PIX = {}
for st, v in TARGETS:
    _, kind, cell = STATS[st]
    k_ = "abs" if kind == "pattern" else kind
    diff = contrast(PS["PROD"][st, v], PS["ERA5"][st, v], k_)
    nz = noise([contrast(HALF[sp, 0][st, v], HALF[sp, 1][st, v], k_) for sp in SPLITS])
    with np.errstate(invalid="ignore", divide="ignore"):
        PIX[st, v] = {"diff": diff, "noise": nz, "t": diff / nz}
    if cell:
        PIX[st, v]["diff_cell"] = contrast(PS["PROD"][cell, v], PS["ERA5"][cell, v], "abs")


def rval(x, reg, kind):
    # x (NP,) -> regional value: area-weighted mean, or area-weighted RMS for the pattern
    ok = REGIONS[reg] & np.isfinite(x)
    if not ok.any():
        return np.nan
    w = WGT[ok]
    if kind == "pattern":
        return float(np.sqrt((w * x[ok] ** 2).sum() / w.sum()))
    return float((w * x[ok]).sum() / w.sum())


ROWS = []
for st, v in TARGETS:
    label, kind, cell = STATS[st]
    for si, s in enumerate(SEASONS):
        for reg in REGIONS:
            e, p = rval(PS["ERA5"][st, v][si], reg, kind), rval(PS["PROD"][st, v][si], reg, kind)
            if kind == "pattern":
                diff = rval(PS["PROD"][st, v][si] - PS["ERA5"][st, v][si], reg, kind)
                D = [rval(HALF[sp, 0][st, v][si] - HALF[sp, 1][st, v][si], reg, kind) for sp in SPLITS]
            else:
                diff = contrast(p, e, kind)
                D = [contrast(rval(HALF[sp, 0][st, v][si], reg, kind), rval(HALF[sp, 1][st, v][si], reg, kind), kind)
                     for sp in SPLITS]
            nz = noise(D)
            tp = PIX[st, v]["t"][si]; ok = REGIONS[reg] & np.isfinite(tp)
            row = dict(var=v, stat=st, statistic=label, season=s, region=reg, ERA5=e, PROD=p, diff=diff, noise=nz,
                       t=diff / nz if nz > 0 else np.nan,
                       pct_pixels_significant=(100 * float((WGT[ok] * (np.abs(tp[ok]) > T_CRIT)).sum() / WGT[ok].sum())
                                               if ok.any() else np.nan))
            if cell:
                row["diff_cell"] = contrast(rval(PS["PROD"][cell, v][si], reg, "abs"), rval(PS["ERA5"][cell, v][si], reg, "abs"), "abs")
                row["gap"] = diff - row["diff_cell"]
            ROWS.append(row)
RES = pd.DataFrame(ROWS)
RES["significant"] = np.abs(RES["t"]) > T_CRIT
RES.to_csv(OUT / f"subgrid_regions_{TAG}.csv", index=False)
print(f"{len(RES)} rows, T_CRIT = {T_CRIT:.2f} -> saved {OUT / f'subgrid_regions_{TAG}.csv'}")
cols = ["var", "stat", "season", "ERA5", "PROD", "diff", "noise", "t", "significant", "pct_pixels_significant", "diff_cell", "gap"]
with pd.option_context("display.width", 240, "display.max_columns", None, "display.float_format", "{:.3f}".format,
                       "display.max_rows", 400):
    display(RES[RES["region"] == "DOMAIN"][cols])

## 4. Maps

One figure per statistic and variable, one row per season: ERA5 | product | difference (relative for the SD) |
t = difference / noise (|t| > `T_CRIT` significant) | for `lag1` and `corr_pair`, the gap = difference at 0.25 deg -
difference of the GCM-cell series. PRUDENCE boxes on the ERA5 maps.

In [ ]:
BORDERS = bu.borders(cfg, "10M")
EXTENT = (PLON.min() - 0.5, PLON.max() + 0.5, PLAT.min() - 0.5, PLAT.max() + 0.5)
MAP_STATS = ["sd_dep", "corr_cell", f"corr_dep_{SHIFTS[0]}", "dep_mean", "lag1", "corr_pair"]


def to_grid(x):
    g = np.full(FLAT.size * FLON.size, np.nan); g[P] = x
    return g.reshape(FLAT.size, FLON.size)


def pct(x, q):
    x = x[np.isfinite(x)]
    return float(np.percentile(x, q)) if x.size else 1.0


def draw(ax, x, title, cmap, vmin, vmax, boxes=False):
    pm = ax.pcolormesh(FLON, FLAT, to_grid(x), shading="nearest", cmap=cmap, vmin=vmin, vmax=vmax)
    bu.draw_borders(ax, BORDERS, lw=0.3)
    if boxes:
        for r, (_, la, lo) in PRUDENCE.items():
            ax.add_patch(Rectangle((lo[0], la[0]), lo[1] - lo[0], la[1] - la[0], fill=False, lw=0.6, ec="k"))
            ax.text(lo[0] + 0.3, la[1] - 1.5, r, fontsize=6, clip_on=True)
    ax.set_xlim(EXTENT[:2]); ax.set_ylim(EXTENT[2:]); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=8)
    plt.colorbar(pm, ax=ax, shrink=0.75)


def map_figure(st, v):
    label, kind, cell = STATS[st]
    ncol = 5 if cell else 4
    fig, axes = plt.subplots(NS, ncol, figsize=(4.3 * ncol, 3.3 * NS), squeeze=False)
    for si, s in enumerate(SEASONS):
        e, p = PS["ERA5"][st, v][si], PS["PROD"][st, v][si]
        diff, t = PIX[st, v]["diff"][si], PIX[st, v]["t"][si]
        lo, hi = pct(np.r_[e, p], 2), pct(np.r_[e, p], 98)
        cmap = "viridis"
        if kind == "pattern":
            hi = max(abs(lo), abs(hi)); lo, cmap = -hi, "RdBu_r"
        draw(axes[si, 0], e, f"{s} ERA5", cmap, lo, hi, boxes=True)
        draw(axes[si, 1], p, f"{s} product", cmap, lo, hi)
        m = pct(np.abs(diff), 98)
        draw(axes[si, 2], diff, f"{s} product {'/ ERA5 - 1' if kind == 'rel' else '- ERA5'}", "RdBu_r", -m, m)
        frac = np.nanmean(np.abs(t) > T_CRIT) if np.isfinite(t).any() else np.nan
        draw(axes[si, 3], t, f"{s} t = difference / noise ({100 * frac:.0f} % of pixels |t| > {T_CRIT:.1f})", "RdBu_r", -6, 6)
        if cell:
            draw(axes[si, 4], diff - PIX[st, v]["diff_cell"][si], f"{s} gap = difference at 0.25 deg - at the GCM cell",
                 "RdBu_r", -m, m)
    fig.suptitle(f"{v}: {label}", fontsize=10)
    plt.tight_layout(); plt.savefig(OUT / f"map_{st}_{v}_{TAG}.png", dpi=90); plt.show(); plt.close(fig)


for st, v in TARGETS:
    if st in MAP_STATS:
        map_figure(st, v)

## 5. Correlation of the departures against distance, per region

ERA5 (black) and product (red): the spatial scale of the structure below the GCM cell. The departures of an
interpolated field are ramps inside each cell that change sign at its edges, so the product curve can be below ERA5 at
short distance and differ in shape.

In [ ]:
for v in VARS:
    fig, axes = plt.subplots(NS, len(REGIONS), figsize=(2.3 * len(REGIONS), 2.0 * NS), squeeze=False, sharex=True, sharey=True)
    d = [DIST[k] for k in SHIFTS]
    for si, s in enumerate(SEASONS):
        for j, reg in enumerate(REGIONS):
            ax = axes[si, j]
            for name, col in (("ERA5", "k"), ("PROD", "C3")):
                ax.plot(d, [rval(PS[name][f"corr_dep_{k}", v][si], reg, "abs") for k in SHIFTS], "o-", color=col, ms=3,
                        label=name)
            ax.axhline(0, c="0.7", lw=0.5); ax.set_ylim(-0.2, 1)
            if si == 0:
                ax.set_title(reg, fontsize=8)
            if j == 0:
                ax.set_ylabel(f"{s}\ncorrelation", fontsize=8)
            if si == NS - 1:
                ax.set_xlabel("km", fontsize=8)
    axes[0, 0].legend(fontsize=6)
    fig.suptitle(f"{v}: correlation of the departures from the GCM-cell mean with the pixel at a distance north", fontsize=10)
    plt.tight_layout(); plt.savefig(OUT / f"departure_correlation_distance_{v}_{TAG}.png", dpi=90); plt.show(); plt.close(fig)

## 6. Summary heatmaps: domain and PRUDENCE regions

Colour: t = difference / noise of the regional value (red: product above ERA5); text: the difference (in % for the SD
of the departure, RMS of the difference for the pattern); bold: significant (|t| > `T_CRIT`).

In [ ]:
HEAT_STATS = ["sd_dep", "corr_cell", f"corr_dep_{SHIFTS[0]}", f"corr_dep_{SHIFTS[2]}", "dep_mean", "lag1", "corr_pair"]
REG_ORDER = list(REGIONS)


def fmt(r):
    kind = STATS[r.stat][1]
    return "" if not np.isfinite(r.diff) else (f"{100 * r.diff:+.0f}%" if kind == "rel" else
                                                f"{r.diff:.2f}" if kind == "pattern" else f"{r.diff:+.2f}")


def heatmap(sub, title, fname):
    sub = sub.copy()
    sub["row"] = sub["var"] + " " + sub["stat"] + " " + sub["season"]
    sub["txt"] = [fmt(r) for r in sub.itertuples()]
    rows = list(dict.fromkeys(sub["row"]))
    T = sub.pivot(index="row", columns="region", values="t").reindex(index=rows, columns=REG_ORDER).values.astype(float)
    L = sub.pivot(index="row", columns="region", values="txt").reindex(index=rows, columns=REG_ORDER).values
    fig, ax = plt.subplots(figsize=(0.95 * len(REG_ORDER) + 3.5, 0.26 * len(rows) + 1.2))
    im = ax.imshow(np.clip(T, -6, 6), cmap="RdBu_r", vmin=-6, vmax=6, aspect="auto")
    for i in range(len(rows)):
        for j in range(len(REG_ORDER)):
            if isinstance(L[i, j], str):
                ax.text(j, i, L[i, j], ha="center", va="center", fontsize=6,
                        fontweight="bold" if np.isfinite(T[i, j]) and abs(T[i, j]) > T_CRIT else "normal")
    ax.set_xticks(range(len(REG_ORDER))); ax.set_xticklabels(REG_ORDER, fontsize=8)
    ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows, fontsize=7)
    plt.colorbar(im, ax=ax, shrink=0.6, label="t = difference / noise")
    ax.set_title(title, fontsize=9)
    plt.tight_layout(); plt.savefig(OUT / fname, dpi=100); plt.show(); plt.close(fig)


for v in VARS:
    heatmap(RES[(RES["var"] == v) & RES["stat"].isin(HEAT_STATS)], f"{v}: product - ERA5 below the GCM cell",
            f"heatmap_{v}_{TAG}.png")
if PAIRS:
    heatmap(RES[RES["stat"] == "corr_pair"], "correlation between variables: product - ERA5", f"heatmap_pairs_{TAG}.png")

## 7. Summary

Per variable and statistic: in how many of the 9 regions x 4 seasons the regional difference is significant, its range,
and the share of the pixels with a significant difference (5 % expected by chance). Deficits below the GCM cell
(`sd_dep` lower, `corr_cell` higher) that are significant and large are those the analogue block could
correct (`corr_dep` differs in either direction); differences of `lag1` / `corr_pair` with a gap close to 0 come from EC-Earth3, not from the downscaling.

In [ ]:
SUM = (RES.groupby(["var", "stat"], sort=False)
          .agg(significant=("significant", "sum"), cases=("significant", "size"), diff_min=("diff", "min"),
               diff_max=("diff", "max"), pct_pixels_significant=("pct_pixels_significant", "mean"))
          .reset_index())
SUM.to_csv(OUT / f"subgrid_summary_{TAG}.csv", index=False)
with pd.option_context("display.width", 200, "display.float_format", "{:.3f}".format, "display.max_rows", 200):
    display(SUM)
print("saved:", OUT / f"subgrid_summary_{TAG}.csv")